In [1]:
import nibabel as nib
from collections import defaultdict
import numpy as np
from brain_connectivity_vizuals import load_mesh_file, create_brain_connectivity_plot, create_modularity_visualization
import pandas as pd
import os
from brain_modularity_pipeline_enhanced_final_v4 import run_enhanced_visualization_pipeline

# Visualize States

In [2]:
# CIMT_rest setup
study = 'CIMT'
study_group = study + '/rs'
ics_path = '../CIMT_data/no_wm/rs/rs_ICA_18c_hptf_ss/rs_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
atlas_path = '../CIMT_data/no_wm/rs/rs_info/rs_01_study_specific_atlas_relabel_no_wm.nii'
mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
roi_coords_path = '../CIMT_data/no_wm/rs/rs_info/resting_atlas_114_mapped_comma.csv'
hmm_run = 'ICA_14c_no_TDE'
k_values = list(range(2, 15))

# # CIMT_task setup
# study = 'CIMT'
# study_group = study + '/LLstim'
# ics_path = '../CIMT_data/no_wm/LLstim/LLstim_ICA_18c_hptf_ss/LLstim_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
# atlas_path = '../CIMT_data/no_wm/LLstim/LLstim_info/LLstim_01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
# roi_coords_path = '../CIMT_data/no_wm/LLstim/LLstim_info/atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_14c_no_TDE'
# k_values = list(range(4, 5))

# # FPI setup
# study_group = 'FPI'
# ics_path = '../FPI_RS_share/no_wm/ICA_13c/ICA_13c_agg__component_ica_.nii'
# atlas_path = '../FPI_RS_share/no_wm/Template/01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../FPI_RS_share/full/Template/MDT_template0.gii'
# roi_coords_path = '../FPI_RS_share/no_wm/FPI_atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_13c_no_TDE'
# k_values = list(range(11, 12))

all_covs, state_plots_paths, modularity_plots_paths = {}, {}, {}
for k in k_values:
    if not os.path.exists(f'{study_group}/results/{hmm_run}/{k}_states/inf_params'):
        raise FileNotFoundError(f"Inference parameters for k={k} states not found")
        
    covs = np.load(open(f'{study_group}/results/{hmm_run}/{k}_states/inf_params/covs_{k}_states.npy', 'rb'))
    print(covs.shape)
    all_covs[k] = covs

    state_plots_path = f'{study_group}/results/{hmm_run}/{k}_states/brain_state_plots'
    os.makedirs(state_plots_path, exist_ok=True)
    state_plots_paths[k] = state_plots_path

    modularity_plots_path = f'{study_group}/results/{hmm_run}/{k}_states/modularity_plots'
    os.makedirs(modularity_plots_path, exist_ok=True)
    modularity_plots_paths[k] = modularity_plots_path

(2, 14, 14)
(3, 14, 14)
(4, 14, 14)
(5, 14, 14)
(6, 14, 14)
(7, 14, 14)
(8, 14, 14)
(9, 14, 14)
(10, 14, 14)
(11, 14, 14)
(12, 14, 14)
(13, 14, 14)
(14, 14, 14)


In [3]:
ics = nib.load(ics_path)
ics_data = ics.get_fdata()
ics_data.shape

(96, 96, 25, 14)

In [4]:
atlas = nib.load(atlas_path)
atlas_data = atlas.get_fdata()
atlas_data.shape

(96, 96, 25)

## Create n_ROI x n_IC Matrix

In [5]:
roi_to_ic = defaultdict(lambda: defaultdict(list)) # roi_idx: {ic: loadings for each coordinate of the roi}
for i in range(ics_data.shape[0]):
    for j in range(ics_data.shape[1]):
        for k in range(ics_data.shape[2]):
            for ic in range(ics_data.shape[3]):
                idx = int(atlas_data[i, j, k])
                if idx > 0:
                    roi_to_ic[idx][ic + 1].append(ics_data[i, j, k, ic])

In [6]:
roi_to_ic[1]

defaultdict(list,
            {1: [np.float64(0.7091153264045715),
              np.float64(0.4090692102909088),
              np.float64(-0.6687657833099365),
              np.float64(-0.911552369594574),
              np.float64(-0.8638740181922913),
              np.float64(-1.3980538845062256),
              np.float64(-1.853228211402893),
              np.float64(-1.9730396270751953),
              np.float64(-1.0620054006576538),
              np.float64(-1.285232424736023),
              np.float64(-1.99537992477417),
              np.float64(-2.111586332321167),
              np.float64(-1.0954887866973877),
              np.float64(-1.285667896270752),
              np.float64(-1.842921495437622),
              np.float64(-1.9304566383361816),
              np.float64(-0.9736332893371582),
              np.float64(-1.0887508392333984),
              np.float64(-1.538509488105774),
              np.float64(-1.237212896347046),
              np.float64(-1.6575136184692383),
    

In [7]:
A = np.zeros((len(roi_to_ic), ics_data.shape[3]))
for roi in sorted(roi_to_ic.keys()):
    for ic in sorted(roi_to_ic[roi].keys()):
        A[roi - 1, ic - 1] = np.mean(roi_to_ic[roi][ic])

A.shape

(114, 14)

## Create n_ROI x n_ROI Correlation Matrix

In [8]:
all_roi_corrs = {}
for k in k_values:
    roi_corrs = []
    for cov in all_covs[k]:
        roi_cov = A @ cov @ A.T
        roi_cov = 0.5 * (roi_cov + roi_cov.T) # enforce symmetry just in case of floating point error
        
        std = np.sqrt(np.diag(roi_cov))
        std_safe = std.copy()
        std_safe[std_safe == 0] = 1
        
        denom = np.outer(std_safe, std_safe)
        roi_corr = roi_cov / denom
        
        std_zeros = (std == 0)
        roi_corr[std_zeros, :] = 0
        roi_corr[:, std_zeros] = 0
        roi_corrs.append(roi_corr)

    print(np.array(roi_corrs).shape)
    all_roi_corrs[k] = roi_corrs

(2, 114, 114)
(3, 114, 114)
(4, 114, 114)
(5, 114, 114)
(6, 114, 114)
(7, 114, 114)
(8, 114, 114)
(9, 114, 114)
(10, 114, 114)
(11, 114, 114)
(12, 114, 114)
(13, 114, 114)
(14, 114, 114)


## Make Plots

In [9]:
vertices, faces = load_mesh_file(mesh_path)

Loading mesh from: ../CIMT_data/no_wm/brain_filled_3_smoothed.gii
  Found faces array with shape: (18857, 3)
  Found vertices array with shape: (9510, 3)
  Successfully loaded mesh: 9510 vertices, 18857 faces


In [10]:
roi_coords_df = pd.read_csv(roi_coords_path)
roi_coords_df.head(20)

,mapped_index,original_index,roi_name,cog_x,cog_y,cog_z,cog_voxel_i,cog_voxel_j,cog_voxel_k
0,1,1,Acumbens_left,-15.355556,66.017901,-20.083333,79.284444,34.933333,142.814321
1,2,2,AID_left,-45.765769,64.444362,-7.542770,103.612615,44.965784,141.555489
2,3,3,AIP_left,-61.894410,29.911610,-21.016484,116.515528,34.186813,113.929288
3,4,4,AIV_left,-38.613395,69.366538,-9.036509,97.890716,43.770793,145.493230
4,5,5,Amygdala_left,-40.170219,15.355125,-37.531890,99.136175,20.974488,102.284100
5,6,8,Au1_left,-67.019196,-0.812283,4.944829,120.615357,54.955864,89.350173
6,7,9,AUD_left,-62.842693,3.178845,12.781300,117.274155,61.225040,92.543076
7,8,10,AuV_left,-69.197761,-2.039801,-6.915423,122.358209,45.467662,88.368159
8,9,11,Brainstem_left,-18.433435,-57.047145,-28.892489,81.746748,27.886009,44.362284
9,10,12,Cent_Gray_left,-7.648063,-22.936525,-0.421196,73.118450,50.663043,71.650780


In [11]:
# Generate brain state plots
all_roi_corrs_thr = {}
for k in k_values:
    roi_corrs_thr = []
    for i in range(len(all_roi_corrs[k])):
        # Visualize top 2.5% of correlations for each state
        R = roi_corrs[i]
        iu = np.triu_indices_from(R, k=1)
        vals = np.abs(R[iu]) # (n_ROI * (n_ROI - 1) / 2,)
        thr = np.percentile(vals, 97.5)
        R_thr = np.zeros_like(R)
        keep = vals >= thr
        R_thr[iu[0][keep], iu[1][keep]] = R[iu[0][keep], iu[1][keep]]
        R_thr = R_thr + R_thr.T # symmetrize
        roi_corrs_thr.append(R_thr)
        create_brain_connectivity_plot(
            vertices=vertices, 
            faces=faces, 
            roi_coords_df=roi_coords_df, 
            connectivity_matrix=R_thr,
            plot_title=f"{study} HMM k={k}, State {i + 1}",
            save_path=os.path.join(state_plots_paths[k], f'state_{i + 1}.html'),
            node_size=8,
            node_color='purple',
            node_border_color='magenta',
            pos_edge_color='red',
            neg_edge_color='blue',
            edge_width=5,
            edge_threshold=0.0,
            mesh_color='lightgray',
            mesh_opacity=0.5,
            label_font_size=8,
            fast_render=False,
        )

    all_roi_corrs_thr[k] = roi_corrs_thr

Creating brain connectivity visualization: CIMT HMM k=2, State 1
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT/rs/results/ICA_14c_no_TDE/2_states/brain_state_plots/state_1.html
Creating brain connectivity visualization: CIMT HMM k=2, State 2
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT/rs/results/ICA_14c_no_TDE/2_states/brain_state_plots/state_2.html
Creating brain connectivity visualization: CIMT HMM k=3, State 1
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT/rs/results/ICA_14c_no_TDE/3_states/brain_state_plots/state_1.html
Creating brain connectivity visualization: CIMT HMM k=3, State 2
Added 114 nodes with valid coordinates
Added 162 edges above threshold 0.0
Saved interactive visualization to: CIMT/rs/results/ICA_14c_no_TDE/3_states/brain_state_plots/state_2.html
Creating bra

In [12]:
for k in k_values:
    np.save(
        f'{study_group}/results/{hmm_run}/{k}_states/inf_params/all_corrs_{k}_states.npy', 
        np.array(all_roi_corrs[k]),
    )
    
    np.savez(
        f'{study_group}/results/{hmm_run}/{k}_states/inf_params/all_corrs_{k}_states.npz', 
        centroids=np.array(all_roi_corrs[k]),
    )

Everything below this requires OMST filtering + expanded modularity pipeline

## Make Plots Using brain_modularity_pipeline_enhanced_final_v4

In [18]:
if not os.path.exists(f'{study}/omst_filter_matrices'):
        raise FileNotFoundError(f"OMST-filtered matrices not found for {study}. Please run OMST filtering first")
    
matrix_paths = {}
modularity_analysis_paths = {}
for k in k_values:
    if study == 'CIMT':
        if study_group == 'CIMT/rs':
            matrix_paths[k] = f'CIMT/omst_filter_matrices/resting_matrices/k{k}_strategy_A_positive.npy'
        elif study_group == 'CIMT/LLstim':
            matrix_paths[k] = f'CIMT/omst_filter_matrices/stim_matrices/k{k}_strategy_A_positive.npy'

    if not os.path.exists(f'{study_group}/results/{hmm_run}/{k}_states/inf_params/modularity_significance_analysis_omst_pos'):
        raise FileNotFoundError(f"Modularity analysis results not found for k={k}. Please run the expanded modularity pipeline first")
        
    modularity_analysis_paths[k] = f'{study_group}/results/{hmm_run}/{k}_states/inf_params/modularity_significance_analysis_omst_pos'

In [19]:
for k in k_values:
    run_enhanced_visualization_pipeline(
        matrix_path=matrix_paths[k],
        netneurotools_results_dir=modularity_analysis_paths[k], 
        mesh_file=mesh_path,
        roi_coords_file=roi_coords_path,
        output_dir=modularity_plots_paths[k] + '/enhanced',
        k_value=k,
    )

ENHANCED BRAIN MODULARITY VISUALIZATION PIPELINE - VERSION 4
With Interactive Camera Controls and Multiple Views
Output directory: CIMT/rs/results/ICA_14c_no_TDE/7_states/modularity_plots/enhanced
Visualization types: ['all', 'intra', 'inter', 'nodes_only', 'significant_only']
Node sizing modes: ['pc', 'zscore', 'both']
Camera views: ['oblique']
Interactive panel: Enabled
Border width: 6 pixels (enhanced visibility)

1. Loading brain mesh and ROI data...
Loading mesh from: ../CIMT_data/no_wm/brain_filled_3_smoothed.gii
  Found faces array with shape: (18857, 3)
  Found vertices array with shape: (9510, 3)
  Successfully loaded mesh: 9510 vertices, 18857 faces
   Loaded 114 ROIs
   Mesh vertices: (9510, 3)
   Mesh faces: (18857, 3)

2. Loading connectivity matrices...
   Loaded matrices: shape (7, 114, 114)

3. Loading NetNeurotools results...
NetNeurotools Loader initialized with base: CIMT/rs/results/ICA_14c_no_TDE/7_states/inf_params/modularity_significance_analysis_omst_pos
   Loadi

{'state1_thresholded_pc_all_oblique': {'state_idx': 0,
  'state_label': 1,
  'threshold': 'thresholded',
  'node_sizing': 'pc',
  'viz_type': 'all',
  'camera_view': 'oblique',
  'n_edges': np.int64(64),
  'Q_total': 0.6134533503841682,
  'Q_z_score': 19.44660778627637,
  'stats': {'total_nodes': 58,
   'total_edges': np.int64(64),
   'n_modules': 5,
   'node_role_distribution': {'Peripheral': 69,
    'Kinless': 32,
    'Ultra-peripheral': 7,
    'Satellite Connector': 6},
   'Q_total': 0.6134533503841682,
   'Q_z_score': 19.44660778627637,
   'camera_view': 'oblique'},
  'path': 'CIMT/rs/results/ICA_14c_no_TDE/7_states/modularity_plots/enhanced/thresholded/node_size_pc/state_1/oblique/all.html'},
 'state1_thresholded_pc_intra_oblique': {'state_idx': 0,
  'state_label': 1,
  'threshold': 'thresholded',
  'node_sizing': 'pc',
  'viz_type': 'intra',
  'camera_view': 'oblique',
  'n_edges': np.int64(64),
  'Q_total': 0.6134533503841682,
  'Q_z_score': 19.44660778627637,
  'stats': {'total